In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 7 · Machine learning: predicting internal circadian phase

*Owner: Part 5 (see issue #5). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 7: "Build a model that classifies, predicts or groups your data." Learning outcome 6: "Build and evaluate machine learning models against proper baselines."*

- [ ] A model that **predicts** each sample's internal circadian phase (`circadian_phase_h`) from expression
- [ ] **Evaluated against proper baselines** with leave-one-subject-out CV (no leakage: scaling and feature selection inside folds)
- [ ] Metric: **median absolute circular error in hours**, with the same folds for every model
- [ ] A fairer "added value" comparison: **clock time + genes vs clock time alone**
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. Matching the clock-time baseline without beating it is an acceptable, reportable result.

## Contract
**Samples:** `data.analysis_samples(for_phase_analysis=True)` (target is `circadian_phase_h`; AF0182 SE dropped; 23 subjects, 214 samples). **Features:** clock genes via `data.load_clock_genes()` (one probe per gene) and/or genes from `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Target:** `circadian_phase_h` encoded as sin/cos. **CV:** `LeaveOneGroupOut`/`GroupKFold` by `subject`. **Metric:** median absolute circular error (h).
**Baselines:** (a) circular mean phase of the training fold; (b) phase predicted from `clock_time_h` only. Optional: add PER3 genotype as a feature.

**Read this before interpreting (data caveat):** `circadian_phase_h` comes in **3-h steps** and is roughly **clock time minus a per-subject offset** (the clock time of each person's melatonin phase 0, ~02:15–07:25 in SE). So baseline (b) will be **strong**: it is only wrong by that offset. Compare the model with it honestly, using the same LOSO folds and the median absolute circular error. **Matching but not beating the clock-time baseline is an acceptable, reportable result.** The fairer question is whether expression adds information beyond clock time: compare **clock time + genes** against **clock time alone** (e.g. does adding genes reduce the per-subject offset error?). Errors below ~1.5 h are not meaningful with 3-h bins. GSE48113 has no phase labels, so there is no external validation set (`data/README.md`).

**Implement in `per3clock.ml`:** `encode_phase`, `decode_phase`, `circular_abs_error_h`, `loso_cv_predict`, `evaluate_phase_models`.

**Outputs:** `results/tables/ml_phase_prediction.csv` (one row per model, including both baselines and clock time + genes); figures `ml_predicted_vs_true_phase.png`, `ml_error_by_model.png`.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 7.1 Features and target

In [ ]:
# TODO(Part 5): build X (genes) and y (sin/cos of circadian_phase_h); groups = subject

## 7.2 Baselines

In [ ]:
# TODO(Part 5): mean-phase and clock-time-only baselines under LOSO

## 7.3 Model

In [ ]:
# TODO(Part 5): e.g. Ridge / ElasticNet / RandomForest in a Pipeline; tune inside folds (nested CV)

## 7.3b Does expression add information beyond clock time?

In [ ]:
# TODO(Part 5): same LOSO folds: clock_time_h only vs clock_time_h + genes; report the difference in median abs circular error (h) with a paired comparison across subjects

## 7.4 Comparison

In [ ]:
# TODO(Part 5): evaluate_phase_models(); table + figures; optional genotype feature

## Interpretation

_TODO(Part 5): does expression predict internal phase better than the baselines, and does it add anything beyond clock time? Which genes carry the signal? If the model only matches the clock-time baseline, say so: that is a valid finding._